In [ ]:
from neo.contexts import Thread, Context
from neo.types import contents as C
from neo.types.roles import Role

# Thread

A Context is a cluster of Contents that are related to each other.   
A Thread is a sequence of Contexts. In this framework, models take in and output a thread.

Content, Context, and Thread are all provider-agnostic. 

In [ ]:
t = Thread(contexts="hi")

In [ ]:
t.display()

In [ ]:
context = Context(
    contents="Hello how can I help you?",
    provider_role=Role.ASSISTANT,
)

In [ ]:
context

In [ ]:
await t.append(context)

In [ ]:
t.display()

In [ ]:
context = Context(
    contents=[
        C.DocumentContent(mime_type="text/plain", data="http://example.com"),
        C.TextContent(data="Summarize the document"),
    ],
    provider_role=Role.USER,
)

In [ ]:
context

In [ ]:
await t.append(context)

## Thread is provider-agnostic and can work with any model/provider

In [ ]:
# Define a simple tool that all models can share
def get_current_time(timezone: str) -> str:
    """Get the current time in a given timezone."""
    from datetime import datetime
    import pytz
    try:
        tz = pytz.timezone(timezone)
        current_time = datetime.now(tz)
        return f"Current time in {timezone}: {current_time.strftime('%Y-%m-%d %H:%M:%S %Z')}"
    except:
        return f"Unknown timezone: {timezone}"

In [ ]:
# Initialize models from different providers
from neo.models.providers.anthropic import AnthropicModel
from neo.models.providers.openai import OpenAIResponseModel
from neo.models.providers.google import GoogleAIModel

# Claude (Anthropic)
claude = AnthropicModel(
    model="claude-3-7-sonnet-20250219",
    instruction="You are Claude, a thoughtful AI assistant.",
    tools=[get_current_time]
)

# GPT (OpenAI)
gpt = OpenAIResponseModel(
    model="gpt-5",
    instruction="You are GPT, a helpful AI assistant.",
    tools=[get_current_time],
    configs={
        "reasoning": {"effort": "low", "summary": "detailed"}, 
        "include": ["reasoning.encrypted_content"], # required for enabled reasoning with multi-turn function calling
    }
)

# Gemini (Google)
gemini = GoogleAIModel(
    model="gemini-2.5-flash",
    instruction="You are Gemini, a creative AI assistant.",
    tools=[get_current_time]
)

In [ ]:
# Start a conversation with Claude
print("🤖 Starting conversation with Claude...")
thread = await claude.acreate("What time is it in Tokyo? And introduce yourself briefly.")

In [ ]:
# Display the thread after Claude's response
thread.display()

In [ ]:
# pass the tool output back
await claude.acreate(user_input=None, base_thread=thread)

In [ ]:
# Now GPT continues the conversation on the same thread
print("\n🤖 GPT joins the conversation...")
await gpt.acreate("What should I do at this time?", base_thread=thread)

In [ ]:
# Display the thread after GPT's response
thread.display()

In [ ]:
# Gemini joins the conversation
print("\n🤖 Gemini joins the conversation...")
await gemini.acreate("Summarize the above conversation.", base_thread=thread)

In [ ]:
thread.display()